In [1]:
%load_ext autoreload
%autoreload 2
import os, glob
import shutil
import pandas as pd
import numpy as np
import otter

In [2]:
ztf_dfs = []
transients_to_skip = {"2019qiz", "AT2020vdq", "AT2018dyk", "AT2022upj"} # these have previously published r-band light curves in OTTER
for f in glob.glob("optical_data/*ztf*csv"):
    transient_name = os.path.basename(f).split("-")[-1].replace(".csv", "")
    if transient_name in transients_to_skip: continue
    df = pd.read_csv(f)
    df["name"] = transient_name
    ztf_dfs.append(df)
ztf = pd.concat(ztf_dfs)
ztf = ztf[ztf.catflags == 0] # from https://web.ipac.caltech.edu/staff/fmasci/ztf/ztf_pipelines_deliverables.pdf

ztf_otter_colmap = {
    "name":"name",
    "mjd":"date",
    "mag":"flux",
    "magerr":"flux_err",
    "filtercode":"filter",
}

ztf = ztf.rename(columns=ztf_otter_colmap)

ztf["filter"] = ztf["filter"].replace({"zg":"g.ztf", "zi":"i.ztf", "zr":"r.ztf"})
ztf["upperlimit"] = ztf.flux_err > np.log(10) / (10 * 2.5)
ztf["flux_unit"] = "mag(AB)"
ztf["filter_eff"] = ztf["filter"].replace({"g.ztf":4746.48,"i.ztf":7829.03,"r.ztf":6366.38}) # From SVO: https://svo2.cab.inta-csic.es/theory/fps/index.php?mode=browse&gname=Palomar&gname2=ZTF&asttype=
ztf["filter_eff_units"] = "AA"
ztf["date_format"] = "mjd"

ztf

,Unnamed: 0,oid,expid,hjd,date,flux,flux_err,catflags,filter,ra,...,clrcounc,exptime,airmass,programid,name,upperlimit,flux_unit,filter_eff,filter_eff_units,date_format
0,0,628115100004852,44835409,2.458203e+06,58202.354097,18.525145,0.041184,0,g.ztf,210.519834,...,0.000032,30.0,1.046,1,SDSS_J1402,False,mag(AB),4746.48,AA,mjd
1,1,628115100004852,44835697,2.458203e+06,58202.356979,18.537483,0.041548,0,g.ztf,210.519842,...,0.000034,30.0,1.041,1,SDSS_J1402,False,mag(AB),4746.48,AA,mjd
2,2,628115100004852,45134172,2.458206e+06,58205.341725,18.499542,0.040441,0,g.ztf,210.519873,...,0.000071,30.0,1.053,1,SDSS_J1402,False,mag(AB),4746.48,AA,mjd
3,3,628115100004852,45136454,2.458206e+06,58205.364549,18.724087,0.047538,0,g.ztf,210.519893,...,0.000081,30.0,1.023,1,SDSS_J1402,False,mag(AB),4746.48,AA,mjd
4,4,628115100004852,45434042,2.458209e+06,58208.340428,18.476143,0.039777,0,g.ztf,210.519963,...,0.000170,30.0,1.042,1,SDSS_J1402,False,mag(AB),4746.48,AA,mjd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
850,850,1520205400000798,158922298,2.459344e+06,59343.222986,17.201595,0.016172,0,r.ztf,191.859893,...,0.000014,30.0,1.105,2,SDSS_J1247,False,mag(AB),6366.38,AA,mjd
851,851,1520205400000798,158922346,2.459344e+06,59343.223461,17.140984,0.015704,0,r.ztf,191.859905,...,0.000014,30.0,1.105,3,SDSS_J1247,False,mag(AB),6366.38,AA,mjd
852,852,1520205400000798,192427107,2.459679e+06,59678.271076,17.228722,0.016391,0,r.ztf,191.859896,...,0.000014,30.0,1.123,2,SDSS_J1247,False,mag(AB),6366.38,AA,mjd
853,853,1520205400000798,195319908,2.459708e+06,59707.199086,17.110275,0.015479,0,r.ztf,191.859890,...,0.000019,30.0,1.115,2,SDSS_J1247,False,mag(AB),6366.38,AA,mjd


In [3]:
ztf.columns

Index(['Unnamed: 0', 'oid', 'expid', 'hjd', 'date', 'flux', 'flux_err',
       'catflags', 'filter', 'ra', 'dec', 'chi', 'sharp', 'filefracday',
       'field', 'ccdid', 'qid', 'limitmag', 'magzp', 'magzprms', 'clrcoeff',
       'clrcounc', 'exptime', 'airmass', 'programid', 'name', 'upperlimit',
       'flux_unit', 'filter_eff', 'filter_eff_units', 'date_format'],
      dtype='str')

In [4]:
radio = pd.read_csv("ecle-radio-photometry.csv", comment="#")
# radio = radio[radio.use_this]

radio

,name,day,date,date_format,array_config,filter,filter_eff,filter_eff_units,flux,flux_err,flux_unit,upperlimit,reducer,source,use_this,comments
0,SDSS_J0748,NaN,57458.91620,mjd,NaN,C (IF 1-8),4.9990,GHz,130.300000,13.5,uJy,False,Kate with pwkit,15B-247,True,NaN
1,SDSS_J0748,NaN,57458.91620,mjd,NaN,C (IF 9-16),7.0990,GHz,95.400000,9.2,uJy,False,Kate with pwkit,15B-247,True,NaN
2,SDSS_J0748,NaN,57458.91620,mjd,NaN,C (all_8bit),5.9570,GHz,112.100000,7.5,uJy,False,Kate with pwkit,15B-247,False,NaN
3,SDSS_J0748,NaN,57914.71674,mjd,NaN,L1,1.2550,GHz,193.400000,73.9,uJy,False,Kate with pwkit,17A-368,True,NaN
4,SDSS_J0748,NaN,57914.71674,mjd,NaN,L2,1.6440,GHz,197.700000,66.1,uJy,False,Kate with pwkit,17A-368,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192,AT2021acak,1999-10-31T00:00:00.000,51482.00000,mjd,NaN,L,1.4351,GHz,0.412227,0.0,mJy,True,NF,FIRST,True,NaN
193,SDSS_J0952,1998-08-31T00:00:00.000,51056.00000,mjd,NaN,L,1.4351,GHz,0.390024,0.0,mJy,True,NF,FIRST,True,NaN
194,SDSS_J1247,2000-01-31T00:00:00.000,51574.00000,mjd,NaN,L,1.4351,GHz,0.420805,0.0,mJy,True,NF,FIRST,True,NaN
195,SDSS_J1241,1997-02-28T00:00:00.000,50507.00000,mjd,NaN,L,1.4351,GHz,0.440643,0.0,mJy,True,NF,FIRST,True,NaN


In [5]:
xrays = pd.read_csv("data/xray/auchettl_xrays_plus_heasarc.csv", comment="#")
xrays

,name,date,date_format,flux,flux_err,upper_err,lower_err,flux_unit,upperlimit,telescope,...,xray_model_param_up_err::Gamma,xray_model_param_lo_err::Gamma,xray_model_param_unit::Gamma,xray_model_param_value::N_H,xray_model_param_unit::N_H,xray_model_param_lo_err::N_H,xray_model_param_up_err::N_H,xray_model_param_upperlimit::N_H,xray_model_param_upperlimit::Gamma,xray_model_reference
0,SDSS_J0938,48200.000000,mjd,6.650000e-14,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
1,SDSS_J0938,52399.000000,mjd,3.480000e-12,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,XMM,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
2,SDSS_J0938,55698.000000,mjd,1.340000e-12,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,XMM,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
3,SDSS_J1055,48365.000000,mjd,1.430000e-14,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
4,SDSS_J1055,48554.000000,mjd,1.730000e-14,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
5,SDSS_J1055,48728.000000,mjd,2.510000e-14,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
6,SDSS_J1055,48955.000000,mjd,1.580000e-14,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
7,SDSS_J1055,49104.000000,mjd,2.450000e-14,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
8,SDSS_J1055,48184.000000,mjd,9.260000e-14,0.000000e+00,0.000000e+00,0.000000e+00,erg/s/cm^2,True,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A
9,SDSS_J1055,48396.000000,mjd,2.590000e-14,2.600000e-14,3.300000e-14,1.900000e-14,erg/s/cm^2,False,ROSAT,...,0.000,0.000,none,NaN,NaN,NaN,NaN,NaN,False,2017ApJ...838..149A


In [6]:
# ,name,date_mjd,filter,filter_eff,flux,flux_err,flux_unit,upperlimit,date,date_format,filter_eff_units,day,array_config,reducer,source,use_this,comments,bibcode

otter_cols = ["name", "date", "date_format", "filter", "filter_eff", "filter_eff_units", "flux", "flux_err", "flux_unit", "upperlimit"]

alldata = pd.concat([ztf[otter_cols], radio[otter_cols], xrays])
alldata.bibcode = alldata.bibcode.fillna("private")

alldata.to_csv("all-photometry.csv")

In [7]:
# now reformat into the OTTER JSONs

overwrite = True
if overwrite:
    shutil.rmtree("private-data")
    db = otter.Otter.from_csvs("ecle-metadata.csv", "all-photometry.csv")
else:
    db = otter.Otter(datadir="private-data")

db

/home/nfranz/research/astro-otter/otter/src/otter/io/otter.py:1331: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  p[key].fillna(0, inplace=True)
/home/nfranz/research/astro-otter/otter/src/otter/io/otter.py:1331: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplac

Attempting to login to https://otter.idies.jhu.edu/api with the following credentials:
username: user-guest
password: test
2019qiz
Adding this as a new object...
AT2017gge
Adding this as a new object...
AT2018bcb
Adding this as a new object...
AT2018dyk
Adding this as a new object...
AT2020vdq
Adding this as a new object...
AT2021acak
Adding this as a new object...
AT2021dms
Adding this as a new object...
AT2021qth
Adding this as a new object...
AT2022fpx
Adding this as a new object...
AT2022upj
Adding this as a new object...
SDSS_J0748
Adding this as a new object...
SDSS_J0807
Adding this as a new object...
SDSS_J0938
Adding this as a new object...
SDSS_J0952
Adding this as a new object...
SDSS_J1055
Adding this as a new object...
SDSS_J1207
Adding this as a new object...
SDSS_J1238
Adding this as a new object...
SDSS_J1241
Adding this as a new object...
SDSS_J1247
Adding this as a new object...
SDSS_J1342
Adding this as a new object...
SDSS_J1350
Adding this as a new object...
SDSS_J

ArangoDB database: otter